In [2]:
pip install transformers datasets torch scikit-learn pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score


In [4]:
# 1. Load CSV Dataset
df = pd.read_csv("IMDB Dataset.csv")  # or "data.csv"

In [5]:
# Convert labels if needed (Kaggle dataset uses text labels)
if df["sentiment"].dtype == "object":
    df["label"] = df["sentiment"].apply(lambda x: 1 if x == "positive" else 0)
    df = df.rename(columns={"review": "text"})
else:
    df = df.rename(columns={"text": "text", "label": "label"})

# Keep only required columns
df = df[["text", "label"]]

# Reduce dataset size for faster training (optional)
df = df.sample(2000, random_state=42)


In [6]:
# 2. Convert to HF Dataset
dataset = Dataset.from_pandas(df)

# Train-test split
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset["train"]
test_dataset = dataset["test"]


In [7]:
# 3. Tokenization
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [8]:
# 4. Format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


In [9]:
# 5. Load Model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# 6. Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.argmax(torch.tensor(logits), dim=1)
    return {"accuracy": accuracy_score(labels, preds)}


In [11]:
# 7. Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    logging_dir="./logs",
)

In [12]:
# 8. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

C:\Users\Pratik\AppData\Local\Temp\ipykernel_14532\3426700272.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [13]:
# 9. Train
trainer.train()


Step,Training Loss


TrainOutput(global_step=400, training_loss=0.3273744964599609, metrics={'train_runtime': 537.5914, 'train_samples_per_second': 5.952, 'train_steps_per_second': 0.744, 'total_flos': 423895675699200.0, 'train_loss': 0.3273744964599609, 'epoch': 2.0})

In [14]:
# 10. Evaluate
results = trainer.evaluate()
print("Evaluation:", results)


Evaluation: {'eval_loss': 0.3445851802825928, 'eval_accuracy': 0.875, 'eval_runtime': 21.8202, 'eval_samples_per_second': 18.332, 'eval_steps_per_second': 2.291, 'epoch': 2.0}


In [15]:
# 11. Save Model
model.save_pretrained("./sentiment_model")
tokenizer.save_pretrained("./sentiment_model")

('./sentiment_model\\tokenizer_config.json',
 './sentiment_model\\special_tokens_map.json',
 './sentiment_model\\vocab.txt',
 './sentiment_model\\added_tokens.json',
 './sentiment_model\\tokenizer.json')

In [17]:
# 12. Custom Prediction
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    # Move inputs to same device as model
    inputs = {key: value.to(device) for key, value in inputs.items()}
    
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()
    
    return "Positive" if pred == 1 else "Negative"

In [18]:
print(predict("This movie was awesome!"))
print(predict("Worst movie ever"))

Positive
Negative


In [19]:
print(predict("This movie was amazing!"))
print(predict("I really hated this film."))
print(predict("It was okay, not great."))

Positive
Negative
Negative


In [23]:
# Take audience input
text = input("Enter a movie review: ")
print("Prediction:", predict(text))

Enter a movie review:  This movie is one of the best movie in world.


Prediction: Positive
